# Campus Voice — かんたん操作画面

無料のColab内で、音声の解析から字幕編集・動画保存まで行います。途中のJSON受け渡しは不要です。

**最初だけ①を実行 → セッションを再起動 → ②を実行してください。** 画面が出たら、セルではなくボタンを使います。

GPU推奨：「ランタイム → ランタイムのタイプを変更 → T4 GPU」。無料枠の割当や処理時間は保証されません。音声はColabへアップロードされます。共有用の公開URLは作りません。

In [ ]:
#@title ① 最初の準備（完了後にセッションを再起動）
"""Install a mutually compatible Colab UI/model stack; stop on failure."""
import subprocess
import sys

REQUIREMENTS = [
    'faster-whisper==1.2.1', 'av==16.0.1',
    'sentence-transformers==6.3.0', 'transformers==5.23.0',
    'huggingface-hub>=1.32,<3', 'scikit-learn', 'ipywidgets==7.8.5',
]


def setup_colab():
    print('1/2: 必要なライブラリを準備しています。', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--upgrade','--quiet',*REQUIREMENTS],check=True)
    print('2/2: 動画・字幕用のツールを準備しています。', flush=True)
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg','fonts-noto-cjk'],check=True)
    print('準備完了。ここで必ず「ランタイム → セッションを再起動」を選び、その後②だけ実行してください。①の再実行は不要です。', flush=True)

setup_colab()


### ② 操作画面を開く
①の後に「ランタイム → セッションを再起動」を行い、このセルの▶を1回押してください。起動後は「素材と解析」「候補と編集」「動画と保存」を上から順に操作します。

**編集途中に②を再実行すると新しい画面になり、作業が引き継がれません。** 終了前に「作業を保存」と「完成MP4を保存」を押してください。再接続後は、元音声と保存した作業JSONを読み込めば再解析せず続けられます。

In [ ]:
#@title ② 操作画面を開く（起動後は下のボタンで操作）
"""Candidate ranking; optional embeddings are computed in Colab, never the editor."""
import math
import re


def summarize_excerpt(segments, max_chars=220):
    """Extract representative sentences; never invent claims or paraphrase ASR.

    Character bigrams work without Japanese tokenizers. Keep selected sentences
    in chronological order; the UI explicitly labels this as an extractive summary.
    """
    sentences = []
    for segment in segments:
        for sentence in re.split(r'(?<=[。！？!?])', segment['text']):
            sentence = sentence.strip()
            if sentence and sentence not in sentences:
                sentences.append(sentence)
    if not sentences:
        return ''
    def grams(text):
        text = re.sub(r'\s+', '', text)
        return set(text[i:i+2] for i in range(len(text)-1))
    features = [grams(t) for t in sentences]
    weights = {}
    for feature in features:
        for gram in feature:
            weights[gram] = weights.get(gram, 0) + 1
    ranking = sorted(range(len(sentences)), key=lambda i: sum(weights[g] for g in features[i])/max(1,len(features[i])) + min(len(sentences[i]),80)/160, reverse=True)
    selected = []
    for i in ranking:
        if any(len(features[i]&features[j])/max(1,len(features[i]|features[j]))>.8 for j in selected):
            continue
        selected.append(i)
        if len(selected)==2:
            break
    text = ' … '.join(sentences[i] for i in sorted(selected))
    return text if len(text)<=max_chars else text[:max_chars-1]+'…'


def suggest_candidates(segments, embeddings=None, limit=10):
    """Rank 30–60s windows with topic, tempo and transcript reaction cues.

    Embeddings must align with segments. Reaction words are not laughter detection.
    Short recordings return fewer candidates rather than duplicating intervals.
    """
    if embeddings is not None and len(embeddings) != len(segments):
        raise ValueError('字幕と埋め込みの件数が一致しません')
    if not segments:
        return []
    boundaries = set()
    if embeddings is not None:
        for i in range(1, len(segments)):
            a, b = embeddings[i - 1], embeddings[i]
            norm = math.sqrt(sum(x*x for x in a) * sum(x*x for x in b))
            similarity = sum(x*y for x, y in zip(a, b)) / norm if norm else 1
            if similarity < .45:
                boundaries.add(i)
    windows = []
    for i, first in enumerate(segments):
        # A long individual subtitle cannot make a 30–60s candidate safely.
        selected = []
        for j in range(i, len(segments)):
            if segments[j]['end'] - first['start'] > 60:
                break
            selected.append(segments[j])
            duration = segments[j]['end'] - first['start']
            if duration < 30:
                continue
            gap = segments[j+1]['start']-segments[j]['end'] if j+1 < len(segments) else 2
            clean_end = gap >= .8 or j+1 in boundaries or j == len(segments)-1
            text = ''.join(s['text'] for s in selected)
            speech = sum(s['end']-s['start'] for s in selected)
            density = min(1, speech / duration)
            tempo = min(1, len(selected) / duration * 5)
            reactions = len(re.findall(r'すごい|面白|おもしろ|まさか|なるほど|[!！]|[（(]笑[)）]', text))
            topic = i in boundaries or i == 0
            score = .4*density + .25*tempo + .15*min(1,reactions/3) + .1*topic + .1*clean_end
            reasons = [f'発話密度 {density:.0%}', f'発話区間 {len(selected)}件']
            if topic and embeddings is not None:
                reasons.append('話題の区切りから開始')
            if reactions:
                reasons.append(f'反応語 {reactions}件（文字情報）')
            if clean_end:
                reasons.append('区切りで終了')
            windows.append(dict(start=first['start'], end=segments[j]['end'],
                                title=first['text'][:40], reason=' / '.join(reasons),
                                score=round(score,4), density=round(density,3),
                                reactions=reactions, topic_start=bool(topic and embeddings is not None),
                                tempo=round(tempo,3), clean_end=clean_end, opening=topic))
    chosen = []
    ranked = sorted(windows, key=lambda c: c['score'], reverse=True)
    def compatible(candidate):
        return not any(max(0,min(candidate['end'], c['end'])-max(candidate['start'],c['start'])) /
                       min(candidate['end']-candidate['start'],c['end']-c['start']) > .25 for c in chosen)
    # Reserve one eligible candidate per time band before filling by quality.
    # Use actual timestamps, including long silences, rather than subtitle count.
    if ranked:
        first = segments[0]['start']
        span = max(1, segments[-1]['end']-first)
        bands = min(5, limit)
        for band in range(bands):
            pool = [c for c in ranked if min(bands-1,int(((c['start']+c['end'])/2-first)/span*bands)) == band]
            candidate = next((c for c in pool if compatible(c)), None)
            if candidate is not None:
                candidate['selection_basis'] = f'解析範囲の第{band+1}/{bands}時間帯から、得点上位の重複しない候補を確保'
                chosen.append(candidate)
    for candidate in ranked:
        if len(chosen) >= limit:
            break
        if compatible(candidate):
            candidate['selection_basis'] = '時間帯別の候補を確保した後、得点順で重複しない候補を追加'
            chosen.append(candidate)
    for candidate in chosen:
        selected = [s for s in segments if s['end']>candidate['start'] and s['start']<candidate['end']]
        candidate['summary'] = summarize_excerpt(selected)
        candidate['summary_method'] = 'extractive-v1'
        candidate['reasons'] = [
            f"発話密度 {candidate['density']:.0%}：無音が少ない区間を加点",
            f"発話テンポ指標 {candidate['tempo']:.0%}：字幕区間数／秒から計算（話者交代数ではありません）",
            candidate['selection_basis'],
        ]
        if candidate['topic_start']:
            candidate['reasons'].append('話題の開始位置と推定されたため加点（先頭または隣接字幕の意味の変化）')
        elif candidate['opening']:
            candidate['reasons'].append('解析範囲の先頭から始まるため加点（意味による判定ではありません）')
        if candidate['reactions']:
            candidate['reasons'].append(f"驚き・反応の文字表現 {candidate['reactions']}件を加点（音声の笑い検出ではありません）")
        if candidate['clean_end']:
            candidate['reasons'].append('無音・話題境界・解析末尾のいずれかで終わるため加点')
    return sorted(chosen, key=lambda c:c['score'], reverse=True)

"""Colab speaker clustering using a public WavLM speaker embedding model.

Labels are estimates per transcript segment, not overlapping-source separation.
"""

def labels_to_segments(segments, labels):
    if len(segments) != len(labels):
        raise ValueError('話者ラベルと字幕の件数が一致しません')
    names = {}
    result = []
    for segment, label in zip(segments, labels):
        if label not in names:
            names[label] = f'話者{len(names)+1}'
        result.append(dict(segment, speaker=names[label]))
    return result


def diarize_segments(audio_path, segments, num_speakers=2):
    """Cluster per-segment embeddings. Requires the expected speaker count.

    Uses at most 6 seconds of each utterance to bound model memory use. Short
    utterances and speaker changes within one subtitle are inherently uncertain.
    """
    import subprocess
    import tempfile
    from pathlib import Path
    import numpy as np
    import torch
    from transformers import AutoFeatureExtractor, AutoModelForAudioXVector
    from sklearn.cluster import AgglomerativeClustering

    if not isinstance(num_speakers, int) or not 1 <= num_speakers <= 10:
        raise ValueError('話者数は1〜10の整数にしてください')
    if len(segments) < num_speakers:
        raise ValueError('字幕数より話者数が多いため、話者数を減らしてください')
    if num_speakers == 1:
        return labels_to_segments(segments, [0]*len(segments))
    model_id = 'microsoft/wavlm-base-plus-sv'
    extractor = AutoFeatureExtractor.from_pretrained(model_id)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    speaker_model = AutoModelForAudioXVector.from_pretrained(model_id).to(device).eval()
    embeddings = []
    # Disk-backed PCM avoids loading an entire 50-minute waveform into RAM.
    with tempfile.TemporaryDirectory() as directory:
        pcm = Path(directory)/'voice.f32'
        subprocess.run(['ffmpeg','-y','-v','error','-i',str(audio_path),'-vn','-ar','16000','-ac','1','-f','f32le',str(pcm)],check=True)
        audio = np.memmap(pcm, dtype='float32', mode='r')
        try:
            for i, segment in enumerate(segments):
                start = int(segment['start']*16000)
                end = min(len(audio), int(segment['end']*16000), start+6*16000)
                if end <= start:
                    raise ValueError('字幕時刻が元音声の範囲を超えています')
                clip = np.array(audio[start:end])
                if len(clip) < 16000:
                    clip = np.pad(clip, (0,16000-len(clip)))
                inputs = extractor(clip, sampling_rate=16000, return_tensors='pt', padding=True)
                with torch.inference_mode():
                    vector = speaker_model(**{k:v.to(device) for k,v in inputs.items()}).embeddings
                    vector = torch.nn.functional.normalize(vector, dim=-1)
                embeddings.append(vector[0].cpu().numpy())
                if (i+1)%25 == 0:
                    print(f'話者解析: {i+1}/{len(segments)}字幕')
        finally:
            del audio
    labels = AgglomerativeClustering(n_clusters=num_speakers, metric='cosine', linkage='average').fit_predict(np.stack(embeddings))
    del speaker_model
    if device == 'cuda':
        torch.cuda.empty_cache()
    return labels_to_segments(segments, labels.tolist())

"""Validate pasted/file edits before replacing the current Colab edit."""
import json
import math


def parse_edit(text, audio_name):
    try:
        edit = json.loads(text)
    except (ValueError, TypeError) as error:
        raise ValueError('編集画面の「Colabへコピー」からコピーしたJSONを貼り付けてください') from error
    if not isinstance(edit, dict):
        raise ValueError('編集JSONが必要です')
    if edit.get('kind') == 'campus-voice-session':
        raise ValueError('作業全体のJSONではなく、編集画面の「Colabへコピー」を使ってください')
    def number(value):
        return isinstance(value, (float, int)) and not isinstance(value, bool) and math.isfinite(value)
    start, end = edit.get('start'), edit.get('end')
    if edit.get('version') != 1 or not number(start) or not number(end) or not 0 <= start < end or end-start > 600:
        raise ValueError('開始・終了を含む編集JSONが必要です（最大600秒）。解析JSONは使えません')
    if edit.get('audio_name') != audio_name:
        raise ValueError('解析時と同じ元音声を選択してください')
    segments = edit.get('segments')
    if not isinstance(segments, list) or not segments:
        raise ValueError('字幕が含まれていません')
    for segment in segments:
        if not isinstance(segment,dict) or not number(segment.get('start')) or not number(segment.get('end')) or not 0 <= segment['start'] < segment['end'] or not isinstance(segment.get('text'),str):
            raise ValueError('字幕の時刻・内容が不正です')
    volume = edit.get('bgm_volume', .12)
    if not number(volume) or not 0 <= volume <= 1:
        raise ValueError('BGM音量は0〜1にしてください')
    if edit.get('format') not in ('vertical','square','landscape'):
        raise ValueError('動画サイズが不正です')
    return edit

"""Shared FFmpeg renderer for the button-based Colab interface."""
import math
import subprocess
import tempfile
from pathlib import Path


def render_clip(audio_path, edit, bgm=None):
    audio_path = Path(audio_path).resolve()
    bgm = str(Path(bgm).resolve()) if bgm else None
    work = Path(tempfile.mkdtemp(prefix='campus-clip-'))
    start,end=float(edit['start']),float(edit['end']); duration=end-start
    assert math.isfinite(duration) and 0<duration<=600
    size={'vertical':(1080,1920),'square':(1080,1080),'landscape':(1920,1080)}[edit['format']]
    def stamp(t):
        cs=max(0,round(t*100)); h,cs=divmod(cs,360000); m,cs=divmod(cs,6000); s,cs=divmod(cs,100)
        return f'{h}:{m:02}:{s:02}.{cs:02}'
    def safe_text(t):
        return str(t).replace('\\','').replace('{','').replace('}','').replace('\n',r'\N').replace('\r','')
    w,h=size
    header=f"""[Script Info]
    ScriptType: v4.00+
    PlayResX: {w}
    PlayResY: {h}
    WrapStyle: 0
    [V4+ Styles]
    Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
    Style: Default,Noto Sans CJK JP,56,&H00FFFFFF,&H00FFFFFF,&H00203930,&H00203930,0,0,0,0,100,100,0,0,1,3,0,5,80,80,60,1
    [Events]
    Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
    """
    lines=[]
    for s in edit['segments']:
        a=max(start,float(s['start'])); b=min(end,float(s['end']))
        if b<=a: continue
        text=(str(s.get('speaker',''))+'：' if s.get('speaker') else '')+str(s['text'])
        lines.append(f'Dialogue: 0,{stamp(a-start)},{stamp(b-start)},Default,,0,0,0,,{safe_text(text)}')
    (work / 'captions.ass').write_text(header+'\n'.join(lines),encoding='utf-8')
    cmd=['ffmpeg','-y','-f','lavfi','-i',f'color=c=0x173e35:s={w}x{h}:r=30:d={duration}', '-ss',str(start),'-t',str(duration),'-i',str(audio_path)]
    if bgm:
        volume=float(edit.get('bgm_volume',0.12)); assert 0<=volume<=1
        cmd+=['-stream_loop','-1','-i',bgm,'-filter_complex',f'[2:a]volume={volume},afade=t=in:d=1,afade=t=out:st={max(0,duration-1)}:d=1[bg];[1:a][bg]amix=inputs=2:duration=first:normalize=0[a]','-map','0:v','-map','[a]']
    else: cmd+=['-map','0:v','-map','1:a']
    cmd+=['-vf','ass=captions.ass','-t',str(duration),'-c:v','libx264','-preset','fast','-threads','2','-pix_fmt','yuv420p','-c:a','aac','-movflags','+faststart','clip.mp4']
    subprocess.run(cmd,check=True,cwd=work,capture_output=True)
    return work / 'clip.mp4'

"""Analysis used by the Colab button UI. No persistent server or external API."""
import gc
import json
import subprocess
import tempfile
from pathlib import Path


def analyze_audio(audio_path, test_seconds=0, speakers=2, use_speakers=True, use_topics=True, count=10, report=print):
    from faster_whisper import WhisperModel
    import torch
    audio_path=Path(audio_path)
    work=Path(tempfile.mkdtemp(prefix='campus-analysis-'))
    source=audio_path
    if test_seconds:
        source=work/'trial.wav'
        subprocess.run(['ffmpeg','-y','-v','error','-i',str(audio_path),'-t',str(test_seconds),'-ar','16000','-ac','1',str(source)],check=True)
    gpu=torch.cuda.is_available()
    report('文字起こしを準備しています。初回はモデルをダウンロードします。')
    model=WhisperModel('small',device='cuda' if gpu else 'cpu',compute_type='float16' if gpu else 'int8')
    segments=[]
    try:
        stream,info=model.transcribe(str(source),language='ja',vad_filter=True,beam_size=5)
        for item in stream:
            if not item.text.strip():continue
            segments.append(dict(start=item.start,end=item.end,text=item.text.strip(),speaker=''))
            (work/'transcript-checkpoint.json').write_text(json.dumps(dict(audio_name=audio_path.name,segments=segments),ensure_ascii=False))
            if len(segments)%25==0:report(f'文字起こし: {item.end/60:.1f}分まで完了')
    except Exception:
        report(f'途中字幕を保存しました: {work / "transcript-checkpoint.json"}')
        raise
    finally:
        del model
        gc.collect()
        if gpu:torch.cuda.empty_cache()
    if not segments:raise ValueError('発話を検出できませんでした。元音声を確認してください。')
    speaker_status=dict(success=False,requested_speakers=speakers)
    if use_speakers:
        report('話者を推定しています。')
        try:
            segments=diarize_segments(source,segments,speakers)
            speaker_status.update(success=True,method='wavlm-utterance-clustering')
        except Exception as error:
            speaker_status['error']=str(error)
            report(f'話者推定を利用できませんでした。名前は手入力できます: {error}')
    embeddings=None
    if use_topics:
        report('話題の区切りを調べています。')
        try:
            from sentence_transformers import SentenceTransformer
            topic_model=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',device='cpu')
            embeddings=topic_model.encode([s['text'] for s in segments],normalize_embeddings=True).tolist()
            del topic_model
        except Exception as error:
            report(f'話題モデルを利用できないため、他の指標で候補を選びます: {error}')
    candidates=suggest_candidates(segments,embeddings,count)
    project=dict(version=1,audio_name=audio_path.name,segments=segments,candidates=candidates,analysis=dict(test_seconds=test_seconds,speaker_diarization=speaker_status['success'],speaker_details=speaker_status,topic_analysis=embeddings is not None,laughter_detection=False,candidate_method='time-balanced-topic-tempo-v3'))
    report(f'完了: {len(segments)}字幕・{len(candidates)}候補。次の「候補と編集」へ進んでください。')
    return project

"""A notebook-local widget interface. No public share link or running API."""
import html
import json
import math
import subprocess
import tempfile
from pathlib import Path


def create_app(upload, download, analyze=analyze_audio, render=render_clip):
    import ipywidgets as w
    from IPython.display import display, Audio, Video, clear_output
    state=dict(audio=None,bgm=None,project=None,video=None,busy=False)
    buttons=[]
    def button(label,action,style=''):
        b=w.Button(description=label,button_style=style,layout=w.Layout(width='auto'))
        buttons.append(b)
        def invoke(_):
            if state['busy']:return
            state['busy']=True
            for item in buttons:item.disabled=True
            with log:
                clear_output(wait=True)
                try:action()
                except Exception as error:print(f'処理できませんでした: {error}\n設定や素材を確認して、同じボタンを押してください。')
                finally:
                    state['busy']=False
                    for item in buttons:item.disabled=False
        b.on_click(invoke)
        return b
    log=w.Output()
    preview=w.Output()
    movie=w.Output()
    info=w.HTML('元音声を選択してください。')
    overview=w.HTML()
    editor=w.VBox()
    speaker_names=w.VBox()
    start=w.FloatText(description='開始（秒）',value=0)
    end=w.FloatText(description='終了（秒）',value=30)
    duration=w.HTML()
    title=w.Text(description='タイトル',layout=w.Layout(width='95%'))
    audio_format=w.Dropdown(description='動画サイズ',options=[('縦 9:16','vertical'),('正方形 1:1','square'),('横 16:9','landscape')])
    volume=w.FloatSlider(description='BGM音量',min=0,max=1,step=.01,value=.12)
    use_bgm=w.Checkbox(description='選択したBGMを使う',value=True)
    candidate=w.Dropdown(description='候補',options=[],layout=w.Layout(width='95%'))
    sort=w.Dropdown(description='並び順',options=[('おすすめ順','score'),('時刻順','time')])
    query=w.Text(description='候補を検索',placeholder='要約・字幕の言葉',layout=w.Layout(width='95%'))
    sample=w.Dropdown(description='解析範囲',options=[('全編（おすすめ）',0),('冒頭3分だけ試す',180)])
    count=w.IntSlider(description='候補数',min=5,max=10,value=10)
    people=w.BoundedIntText(description='出演人数',min=1,max=10,value=2)
    diarize=w.Checkbox(description='話者を自動推定する',value=True)
    topics=w.Checkbox(description='話題の区切りも使う',value=True)
    def refresh_info():
        project=state.get('project')
        scope=''
        if project:
            seconds=project.get('analysis',{}).get('test_seconds')
            scope=' ／ 解析範囲：'+('全編' if seconds==0 else f'冒頭{seconds}秒のみ' if seconds else '不明')
        info.value=f"元音声：{html.escape(state['audio'].name) if state['audio'] else '未選択'} ／ BGM：{html.escape(state['bgm'].name) if state['bgm'] else 'なし'}"+scope
    def choose_audio():
        if state['project'] and not reanalyze.value:
            raise ValueError('先に作業を保存し、「再解析を許可」にチェックしてから別の元音声を選んでください')
        paths=upload_one()
        state.update(audio=paths,project=None,video=None)
        candidate.options=[];editor.children=[];speaker_names.children=[];overview.value=''
        with preview:clear_output()
        with movie:clear_output()
        refresh_info();print('元音声を選びました。「解析する」を押してください。')
    def upload_one():
        data=upload()
        names=[Path(n) for n in data if Path(n).suffix.lower() in {'.mp3','.wav','.m4a','.ogg','.flac'}]
        if len(names)!=1:raise ValueError('音声ファイルを1本だけ選択してください')
        # Colab writes bytes to cwd; use our own unique directory to avoid name collisions.
        directory=Path(tempfile.mkdtemp(prefix='campus-upload-'))
        path=directory/names[0].name
        path.write_bytes(data[str(names[0])])
        return path
    def choose_bgm():
        state['bgm']=upload_one();refresh_info();print('BGMを選びました。解析のやり直しは不要です。')
    def remove_bgm():
        state['bgm']=None;refresh_info();print('BGMなしにしました。')
    def run_analysis():
        if not state['audio']:raise ValueError('先に元音声を選択してください')
        if state['project'] and not reanalyze.value:
            raise ValueError('すでに解析済みです。再解析すると編集を破棄します。「再解析を許可」にチェックしてください。')
        project=analyze(state['audio'],test_seconds=sample.value,speakers=people.value,use_speakers=diarize.value,use_topics=topics.value,count=count.value,report=print)
        state['project']=project;state['video']=None;reanalyze.value=False
        refresh_info();refresh_candidates();refresh_names()
        if not project['candidates']:
            start.value=project['segments'][0]['start'];end.value=min(project['segments'][-1]['end'],start.value+60)
            overview.value='30〜60秒の候補がないため、範囲を手動指定してください。';refresh_segments()
    reanalyze=w.Checkbox(description='再解析を許可（字幕・範囲の編集を破棄）',value=False)
    def refresh_segments(*_):
        duration.value=f'<b>選択時間：{end.value-start.value:.1f}秒</b>（おすすめ30〜60秒）'
        if not state['project']:return
        rows=[]
        for s in state['project']['segments']:
            if s['end']<=start.value or s['start']>=end.value:continue
            name=w.Text(value=s.get('speaker',''),description='話者名')
            text=w.Textarea(value=s['text'],description='字幕',layout=w.Layout(width='95%',height='80px'))
            name.observe(lambda change,segment=s:segment.update(speaker=change['new']),names='value')
            text.observe(lambda change,segment=s:segment.update(text=change['new']),names='value')
            rows.append(w.VBox([w.HTML(f"{s['start']:.1f}〜{s['end']:.1f}秒"),name,text]))
        editor.children=rows
    def refresh_names():
        if not state['project']:return
        rows=[]
        for name in dict.fromkeys(s.get('speaker','') for s in state['project']['segments']):
            if not name:continue
            field=w.Text(value=name,description=name)
            def rename(old=name,field=field):
                new=field.value.strip()
                if not new:raise ValueError('話者名を入力してください')
                for s in state['project']['segments']:
                    if s.get('speaker')==old:s['speaker']=new
                refresh_names();refresh_segments();print('話者名を変更しました。')
            rows.append(w.HBox([field,button('名前をまとめて変更',rename)]))
        speaker_names.children=rows
    def select_candidate(change):
        index=change['new']
        if index is None or not state['project']:return
        c=state['project']['candidates'][index]
        start.value=c['start'];end.value=c['end'];title.value=c.get('title','')
        reasons=c.get('reasons',[c.get('reason','理由の記録なし')])
        overview.value='<b>内容の要約（字幕の抜粋）</b><p>'+html.escape(c.get('summary','このデータには要約がありません'))+'</p><b>選出理由</b><ul>'+''.join('<li>'+html.escape(r)+'</li>' for r in reasons)+'</ul><small>要約・理由は解析時点の情報です。修正後も自動更新されません。</small>'
        refresh_segments()
    def refresh_candidates(*_):
        if not state['project']:return
        project=state['project'];indices=list(range(len(project['candidates'])))
        q=query.value.strip().lower()
        if q:
            indices=[i for i in indices if q in (' '.join(str(project['candidates'][i].get(k,'')) for k in ['title','summary','reason'])+' '+' '.join(s['text'] for s in project['segments'] if s['end']>project['candidates'][i]['start'] and s['start']<project['candidates'][i]['end'])).lower()]
        indices.sort(key=lambda i:project['candidates'][i]['start'] if sort.value=='time' else -project['candidates'][i].get('score',0))
        candidate.options=[(f"{project['candidates'][i]['start']/60:.1f}分〜 ／ {project['candidates'][i].get('title','候補')}",i) for i in indices]
        if indices:select_candidate(dict(new=indices[0]))
        else:overview.value='一致する候補がありません。検索を変えるか、範囲を手動指定してください。'
    def current_edit():
        if not state['project'] or not state['audio']:raise ValueError('先に解析または保存した作業の読み込みをしてください')
        project=state['project']
        edit=dict(version=1,audio_name=project['audio_name'],start=start.value,end=end.value,title=title.value,format=audio_format.value,bgm_volume=volume.value,segments=[s for s in project['segments'] if s['end']>start.value and s['start']<end.value])
        edit=parse_edit(json.dumps(edit),state['audio'].name)
        duration_audio=float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',str(state['audio'])],check=True,capture_output=True,text=True).stdout)
        if end.value>duration_audio+.1:raise ValueError('選択範囲が元音声の長さを超えています')
        return edit
    def listen():
        edit=current_edit();a=max(0,edit['start']-5);length=edit['end']+5-a
        path=Path(tempfile.mkdtemp(prefix='campus-preview-'))/'preview.wav'
        subprocess.run(['ffmpeg','-y','-v','error','-ss',str(a),'-i',str(state['audio']),'-t',str(length),'-ar','16000','-ac','1',str(path)],check=True)
        with preview:clear_output();display(Audio(filename=str(path)))
        print('選択範囲の前後5秒を含む試聴です。')
    def make_video():
        edit=current_edit();state['video']=None
        with movie:clear_output()
        print('動画を作成しています。完了するまでお待ちください。')
        path=render(state['audio'],edit,state['bgm'] if use_bgm.value else None)
        state['video']=Path(path)
        state['video_edit']=json.dumps(edit,ensure_ascii=False)
        state['video_bgm']=str(state['bgm']) if use_bgm.value else None
        with movie:display(Video(filename=str(path),embed=True,width=300))
        print('完成しました。「完成MP4を保存」を押してください。')
    def save_video():
        if not state['video']:raise ValueError('先に「動画を作る」を押してください')
        if json.dumps(current_edit(),ensure_ascii=False)!=state.get('video_edit') or (str(state['bgm']) if use_bgm.value else None)!=state.get('video_bgm'):
            raise ValueError('編集内容やBGMが変わっています。「動画を作る」で作り直してから保存してください')
        download(str(state['video']))
    def save_session():
        edit=current_edit()
        path=Path(tempfile.mkdtemp(prefix='campus-session-'))/'campus-voice-session.json'
        path.write_text(json.dumps(dict(kind='campus-voice-session',version=1,project=state['project'],selection=edit),ensure_ascii=False,indent=2))
        download(str(path));print('作業を保存しました。次回は元音声とこのJSONを読み込めます。')
    def restore_session():
        if not state['audio']:raise ValueError('先に元音声を選んでください')
        uploaded=upload()
        if len(uploaded)!=1:raise ValueError('作業JSONを1つ選択してください')
        data=json.loads(next(iter(uploaded.values())))
        if data.get('kind')!='campus-voice-session':raise ValueError('「作業を保存」で取得したJSONを選択してください')
        project=data['project'];saved=data['selection']
        parse_edit(json.dumps(saved),state['audio'].name)
        if project.get('audio_name')!=state['audio'].name or not isinstance(project.get('segments'),list) or not isinstance(project.get('candidates'),list):raise ValueError('作業JSONか元音声が一致しません')
        state['project']=project;state['video']=None
        refresh_candidates();refresh_names()
        start.value=saved['start'];end.value=saved['end'];title.value=saved.get('title','');volume.value=saved.get('bgm_volume',.12);audio_format.value=saved['format']
        refresh_info();refresh_segments();print('作業を再開しました。解析のやり直しは不要です。')
    candidate.observe(select_candidate,names='value')
    query.observe(refresh_candidates,names='value');sort.observe(refresh_candidates,names='value')
    start.observe(refresh_segments,names='value');end.observe(refresh_segments,names='value')
    sections=[
        w.VBox([w.HTML('<h3>1. 音声を選んで解析</h3><p>元のラジオ音声を選び「解析する」を押します。全編は時間がかかります。</p>'),button('元音声を選ぶ',choose_audio),info,w.HBox([button('BGMを選ぶ（任意）',choose_bgm),button('BGMなしにする',remove_bgm)]),sample,people,diarize,topics,count,reanalyze,button('解析する',run_analysis,'primary'),button('保存した作業を読み込む',restore_session)]),
        w.VBox([w.HTML('<h3>2. 候補を確認して編集</h3><p>候補を選び、前後を試聴。字幕・話者名・範囲を修正します。変更はこの画面内で保持されます。</p>'),query,sort,candidate,overview,w.HBox([start,end]),duration,title,button('前後を試聴',listen),preview,w.HTML('<b>話者名をまとめて設定</b>'),speaker_names,editor,button('作業を保存（次回再開用）',save_session)]),
        w.VBox([w.HTML('<h3>3. 動画を作って保存</h3><p>現在の選択範囲と字幕で作ります。2本目は「候補と編集」へ戻って選び直してください。</p>'),audio_format,volume,use_bgm,button('動画を作る',make_video,'success'),movie,button('完成MP4を保存',save_video),button('作業を保存（次回再開用）',save_session)])
    ]
    app=w.VBox([w.HTML('<h2>Campus Voice</h2><p>①素材と解析 → ②候補と編集 → ③動画と保存</p><p>終了前に「作業を保存」。Colabの接続が切れると未保存の作業は消えることがあります。</p>'),w.HTML('<b>進行状況・メッセージ</b>'),log,*[control for section in sections for control in section.children]])
    return app


def launch_colab_app():
    print('② 起動準備：Colabの表示機能を読み込んでいます。', flush=True)
    from google.colab import files, output
    from IPython.display import display, HTML
    import ipywidgets as w
    from importlib.metadata import version
    if w.__version__ != version('ipywidgets'):
        raise RuntimeError('古い表示ライブラリがメモリに残っています。「ランタイム → セッションを再起動」の後、②だけ実行してください。')
    print(f'表示ライブラリ: ipywidgets {w.__version__}', flush=True)
    output.enable_custom_widget_manager()
    display(HTML('<p><b>操作画面を表示しています。</b>「元音声を選ぶ」が出ない場合は出力欄を展開してください。それでも出ない場合は、このメッセージと②の出力を送ってください。</p>'))
    app=create_app(files.upload,files.download)
    status=w.HTML('最初に「表示確認」を押してください。')
    check=w.Button(description='表示確認',button_style='info')
    def confirm(_):
        status.value='✓ 操作できます。下の「元音声を選ぶ」から始めてください。'
    check.on_click(confirm)
    display(w.VBox([w.HBox([check,status]),app]))
    print('② 起動処理が完了しました。上の「表示確認」を押し、「元音声を選ぶ」から進めてください。', flush=True)
    return app

campus_app = launch_colab_app()


### よくある困りごと
- ボタンが出ない：②の「起動準備」等の文字が表示されるか確認してください。出力が折りたたまれていたら展開し、再起動後に②を実行してください。Colabからウィジェット表示の許可を求められたら確認してください。
- 解析が長い：全編は時間がかかります。進行状況欄を確認してください。最初の確認だけなら解析範囲を「冒頭3分」にできます。
- HF Hubの未認証の警告：ダウンロードが進んでいれば待って構いません。
- 話者名が違う：「候補と編集」で名前を直してください。自動推定は同時発話・短い相づち等で誤ることがあります。
- 候補の要約は字幕の抜粋です。内容と範囲は必ず試聴して確認してください。
- Colabが切れた：①②で起動し直し、元音声→保存した作業の順に読み込みます。BGMは別途選び直します。
- データの削除：手元の保存ファイルを削除し、Colabは「接続を解除して削除」で一時データを削除できます。Driveへの自動保存はしません。